**Table of contents**<a id='toc0_'></a>    
- [Pivot Tables for the Paper](#toc1_)    
- [Let's Create Summary](#toc2_)    
- [Pivot Views](#toc3_)    
  - [Structural Text Elements](#toc3_1_)    
  - [Math](#toc3_2_)    
- [Categorical Groups](#toc4_)    
  - [Morphological](#toc4_1_)    
  - [LAnguage Contact](#toc4_2_)    
- [Orthography](#toc5_)    
  - [Input Medium](#toc5_1_)    
  - [Diacritics](#toc5_2_)    
  - [Register Style](#toc5_3_)    
  - [Morph](#toc5_4_)    
- [Noise Cuated](#toc6_)    
- [Grammar](#toc7_)    
- [Linguistic Variety](#toc8_)    
- [Structural Text](#toc9_)    
    - [Math styling](#toc9_1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Pivot Tables for the Paper](#toc0_)



In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import math
from pathlib import Path

from xarch_tokenizers.logging.report_utils import (
    load_predictions,
    load_all_samples,
    clean_model_name,
)
from xarch_tokenizers.logging.plot_utils import (
    setup_styles,
    get_new_6_fig,
    MODEL_TO_COLOR,
    get_new_fig,
)

from utils import *
from notebook_views import *

setup_styles()
%load_ext autoreload
%autoreload 2

# <a id='toc2_'></a>[Let's Create Summary](#toc0_)

In [3]:
OUTPUT_DIR = Path("./output/results-v5")
OUTPUT_DIR_ = OUTPUT_DIR / "summary"
OUTPUT_DIR_ = OUTPUT_DIR / "summary-rebuttal"
METRIC = "acc_norm"
# read all samples
base_dir = Path("../results/paper-v5").resolve().absolute()

# # keeps set_id, model pairs that are correct for canonical
# suffix = "(Still correct)"
# only_keep_canonical_true = True
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

# suffix = "(Remains or Becomes Correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = True
# only_keep_perturbed_true = False

# suffix = "(Flipped to correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = True

suffix = "(All)"
only_keep_canonical_true = False
keep_remains_or_becomes_correct = False
only_keep_perturbed_true = False

code_pattern = "*"
title_pattern = "All Datasets"
OUTPUT_DIR = OUTPUT_DIR_ / "all"

OUTPUT_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
base_dir.exists()
pred_files = list([p for p in base_dir.rglob(f"samples{code_pattern}.jsonl")])
len(pred_files), pred_files[:5]


(2528,
 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_general_currency_symbol_2025-09-20T10-09-51.814046.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/s

In [4]:
## Read all samples into a DataFrame
## exclude general dataset for now
all_samples = load_all_samples(
    base_dir,
    patterns=[code_pattern],
    exclude_patterns=["general"],
    flatten_doc=True,
    match_date=False,
    simplify_df=True,
)

2366 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_grammatical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_t

We want a summary table like below

| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | BLOOM        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | ByT5         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Comma        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | GPT-2        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.823529 |
| no_filter        | GPT-4o       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        1        |             0.941176 |
| no_filter        | Gemma-2      | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | Llama-3.2    | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.882353 |
| no_filter        | Phi-3        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Qwen-3       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Tekken       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.176471 |  0.392953 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | TokenMonster | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.764706 |
| no_filter        | XGLM         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.823529 |       0.392953 | 0.882353 |  0.332106 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | mBERT        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.764706 |


started with this

|          |                     |            |     |          |                   |                     |             |                  |                     |                     |
| -------- | ------------------- | ---------- | --- | -------- | ----------------- | ------------------- | ----------- | ---------------- | ------------------- | ------------------- |
| Category | Perturbation (Task) | Model name | acc | acc_norm | number of samples | filter              | acc_std_err | acc_norm_std_err | number of canonical | number of perturbed |
|          | romanization        | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | romanization        | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_canonical      |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_perturbed      |             |                  |                     |                     |								

In [5]:
import warnings

from category_mapping import SUBCATEGORY_TO_CATEGORY
from typing import List

FILTERS = [
    "no_filter",
    "only_canonical_correct",
    "canonical_wrong_at_least_one_perturb_correct",
    "remains_or_becomes_correct",
]
keys = {
    "no_filter": (False, False, False),
    "only_canonical_correct": (True, False, False),
    "canonical_wrong_at_least_one_perturb_correct": (False, False, True),
    "remains_or_becomes_correct": (False, True, False),
}


all_summaries = get_summaries(all_samples, subcategories=None)

N duplicates: 30604
N duplicates: 61193
N duplicates: 21964
N duplicates: 63314


In [6]:
## sanity check
print(
    all_summaries[all_summaries["task_pretty_name"] == "Fullwidth Characters"]
    .head(n=10)
    .to_markdown(index=False)
)


| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm | group_name                           |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|:-------------------------------------|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | toke

In [7]:
save_path = OUTPUT_DIR / "summary.tsv"

all_summaries.to_csv(save_path, sep="\t")
print(f"File saved at \n{save_path}")

File saved at 
output/results-v5/summary-rebuttal/all/summary.tsv


# <a id='toc3_'></a>[Pivot Views](#toc0_)

FILTER

In [8]:
FILTER = "no_filter"

In [9]:
style_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & ~(all_summaries["task"].str.contains("math"))
    & ~(all_summaries["task"].str.contains("stem"))
    # & all_summaries["task_pretty_name"].isin(styling_categories)
]
styled_df = get_styled_df(style_summaries, columns=["task_pretty_name", "langs"])
# styled_df = get_styled_df(style_summaries, index=["task_pretty_name", "langs"], columns=["model_name"])
styled_df

/Users/gsaltintas/code/phd/tokenizers/notebooks/notebook_views.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# Main Table

In [46]:


FILTER = "no_filter"

# Define categories for the LaTeX table
latex_table_categories = {
    "Input (Non-EN)": [
        "English keyboard",
        "Arabic Keyboard for Farsi", 
        "Number Romanization",
        "Romanization",
        # "Partially romanized",
        "Traditional"
    ],
    "Diacritics (Non-EN)": ["Optional diacritics"],
    "Orthographic Errors (EN)": [
        "Orthographic errors",
        "Grammatical errors"
    ],
    "Orthographic Errors (Non-EN)": [
        "Orthographic errors",
        "Grammatical errors"
    ],
    "Morphological (EN)": [
    "Contractions",
    "Derivations",
    "Inflections",],
    "Morphological (Non-EN)": [
    "Contractions",
    "Derivations",
    "Inflections",],
    "Noise (EN)": [
        "Homoglyphs", "Plausible diacritics errors", "Keyboard proximity errors",
        "OCR Errors", "Character deletion", "Space removal", "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space",
    ],
    "Noise (Non-EN)": [
        "Homoglyphs", "Plausible diacritics errors", "Keyboard proximity errors", 
        "OCR Errors", "Character deletion", "Space removal", "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space",
    ],
    "LaTeX": ["LaTeX"],
    "Math/STEM (EN)": [
        # "Spelled out", 
        "Unusual formatting"
        ],
    # "Math/STEM (Non-EN)": ["Turkish", "Italian", "Chinese", "Farsi"],
    "Unicode": [
        "Fullwidth Characters", "Decorative Unicode", "Scripted text",
        "Double struck", "Enclosed Characters", "Unicode formatting",
    ],
}

# whitespace_robustness_dict = {
#     # These would need to be calculated as robustness drops from your whitespace analysis
#     # For now using placeholder - you'd calculate (canonical_whitespace - perturbed_whitespace) / canonical_whitespace
#     "Aya": 0.15,  # Example: 15% drop
#     "BLOOM": 0.18,
#     "ByT5": 0.12,
#     "Comma": 0.14,
#     "GPT-2": 0.16,
#     "GPT-4o": 0.13,
#     "Gemma-2": 0.17,
#     "Llama-3.2": 0.19,
#     "Phi-3": 0.14,
#     "Qwen-3": 0.16,
#     "Tekken": 0.15,
#     "TokenMonster": 0.22,
#     "XGLM": 0.08,
#     "mBERT": 0.09
# }

if __name__ == "__main__":
    # You need to define your canonical_by_task dictionary here
    
    # Generate robustness data and save to JSONL
    canonical_by_task = get_canonical_by_task(all_summaries, FILTER)
    robustness_data = generate_robustness_latex_table_dict(all_summaries, canonical_by_task, FILTER, latex_table_categories=latex_table_categories, group_langs=True)
    jsonl_file = save_to_jsonl(robustness_data, "robustness_results.jsonl")
    
    # Generate robustness LaTeX table
    robustness_latex = load_and_generate_robustness_latex_table(jsonl_file)
    
    print("Robustness LaTeX Table Generated!")
    print("=" * 60)
    print(robustness_latex)
    
    # Save to file
    with open("robustness_tokenization_table.tex", "w") as f:
        f.write(robustness_latex)
    print("\nRobustness table saved to: robustness_tokenization_table.tex")
    get_robustness_styled_df(jsonl_file)


Processing model: Aya, category: Input (Non-EN), data size: 7, perturbations: ['English keyboard', 'Arabic Keyboard for Farsi', 'Number Romanization', 'Romanization', 'Traditional'], 
	tasks: ['tokenizer_robustness_completion_italian_english_keyboard'
 'tokenizer_robustness_completion_farsi_romanization'
 'tokenizer_robustness_completion_chinese_traditional'
 'tokenizer_robustness_completion_farsi_arabic_keyboard_for_farsi'
 'tokenizer_robustness_completion_turkish_english_keyboard'
 'tokenizer_robustness_completion_chinese_romanization'
 'tokenizer_robustness_completion_farsi_number_romanization']
Processing model: Aya, category: Diacritics (Non-EN), data size: 2, perturbations: ['Optional diacritics'], 
	tasks: ['tokenizer_robustness_completion_chinese_optional_diacritics'
 'tokenizer_robustness_completion_farsi_optional_diacritics']
Processing model: Aya, category: Orthographic Errors (EN), data size: 2, perturbations: ['Orthographic errors', 'Grammatical errors'], 
	tasks: ['tokeni

In [48]:
get_robustness_styled_df(jsonl_file)

,Input (Non-EN),Diacritics (Non-EN),Orthographic Errors (EN),Orthographic Errors (Non-EN),Morphological (EN),Morphological (Non-EN),Noise (EN),Noise (Non-EN),LaTeX,Math/STEM (EN),Unicode,Average
model_name,,,,,,,,,,,,
TokenMonster,0.23,0.33,0.09,0.02,0.23,-0.05,0.11,0.19,0.23,0.11,0.52,0.18
XGLM,0.35,0.49,0.10,0.12,0.25,0.07,0.12,0.22,0.30,0.29,0.12,0.22
BLOOM,0.31,0.35,0.13,0.08,0.18,0.11,0.18,0.19,0.25,0.11,0.57,0.22
Comma,0.29,0.43,0.05,0.07,0.18,0.00,0.11,0.21,0.23,0.29,0.61,0.23
ByT5,0.30,0.44,0.04,0.06,0.27,0.05,0.14,0.18,0.18,0.29,0.53,0.23
mBERT,0.33,0.44,0.11,0.11,0.23,0.06,0.18,0.22,0.15,0.22,0.62,0.24
GPT-4o,0.30,0.52,0.08,0.05,0.21,0.06,0.16,0.20,0.25,0.33,0.55,0.25
GPT-2,0.35,0.46,0.07,0.10,0.25,0.06,0.15,0.21,0.25,0.35,0.53,0.25
Phi-3,0.33,0.46,0.16,0.09,0.27,0.08,0.17,0.21,0.25,0.22,0.55,0.25
